**Assignment 1: Understanding Apache Spark and Distributed Processing**\
**25MML1001 Karthik M**

# Theory

Q. Explain the concept of distributed data processing. How does Apache Spark improve performance compared to traditional data processing systems? Include the role of in-memory computation.

## What is Distributed Data Processing?

Distributed data processing means splitting a large dataset across multiple machines (nodes) and processing them in parallel, rather than on a single machine. Each node works on its own chunk of data simultaneously, and the results are combined at the end.

Think of it like this — instead of one person reading a 1000-page book, you give 100 pages each to 10 people. The work finishes 10x faster.

---

## How Apache Spark Improves Performance

Traditional systems like **Hadoop MapReduce** process data by reading from disk, processing, writing back to disk, and repeating this for every stage. This constant disk I/O makes it extremely slow, especially for multi-step jobs.

Apache Spark improves on this in several ways:

| Feature | Hadoop MapReduce | Apache Spark |
|---|---|---|
| Storage | Reads/writes to disk after every step | Keeps intermediate data in RAM |
| Speed | Slow due to repeated disk I/O | Up to 100x faster for iterative jobs |
| Ease of use | Verbose Java code | Clean Python / Scala / SQL APIs |
| Real-time support | Batch only | Batch + Streaming |
| Fault tolerance | Replication-based | RDD lineage-based recomputation |
| Processing model | Two-stage (Map + Reduce only) | Multi-stage DAG execution |

---

## Role of In-Memory Computation

In-memory computation is the core reason Spark is fast. Instead of writing intermediate results to disk between stages, Spark stores them in **RAM across the cluster**.

This matters most for:

- **Iterative algorithms** – Machine learning models train by passing over the same data dozens or hundreds of times. With Hadoop, every pass hits the disk. With Spark, the data stays in memory and each pass is near-instant.
- **Interactive queries** – Analysts running repeated queries on the same dataset benefit hugely since the data is already loaded in memory.
- **Multi-stage pipelines** – Complex ETL jobs with many transformation steps avoid the disk read/write penalty at every stage.

### How Spark manages in-memory data:

- Data is stored as **RDDs (Resilient Distributed Datasets)** or **DataFrames** partitioned across nodes in the cluster
- Spark uses a **DAG (Directed Acyclic Graph)** scheduler to plan the most efficient execution path before running anything
- If a node fails, Spark does **not** need to restart the entire job — it recomputes only the lost partition using the **lineage graph** (the recorded history of transformations)
- If data doesn't fit in RAM, Spark **spills to disk gracefully** rather than crashing

# Practical

Q. Write a PySpark program to:
- Create a DataFrame with at least 5 records (columns: ID, Name, Age, Salary)
- Display the schema
- Show all records
- Filter employees with salary greater than 30,000

In [1]:
# Import libraries
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, FloatType

In [2]:
# Initiate Spark Session
spark = SparkSession.builder \
    .appName("DA1") \
    .getOrCreate()

In [3]:
# Define schema
schema = StructType([
    StructField("ID", IntegerType(), False),
    StructField("Name", StringType(), False),
    StructField("Age", IntegerType(), True),
    StructField("Salary", FloatType(), True)
])

In [4]:
# Manually create data with 5+ records
data = [
    (1, "Karthik", 22, 45000.0),
    (2, "Priya",   25, 28000.0),
    (3, "Arjun",   30, 62000.0),
    (4, "Meena",   27, 31000.0),
    (5, "Ravi",    35, 75000.0),
    (6, "Divya",   23, 22000.0),
    (7, "Suresh",  40, 90000.0)
]

In [5]:
# Create DataFrame
df = spark.createDataFrame(data, schema=schema)

In [6]:
# Display schema
print("Schema:")
df.printSchema()

Schema:
root
 |-- ID: integer (nullable = false)
 |-- Name: string (nullable = false)
 |-- Age: integer (nullable = true)
 |-- Salary: float (nullable = true)



In [7]:
# Show all records
print("All Records:")
df.show()

All Records:
+---+-------+---+-------+
| ID|   Name|Age| Salary|
+---+-------+---+-------+
|  1|Karthik| 22|45000.0|
|  2|  Priya| 25|28000.0|
|  3|  Arjun| 30|62000.0|
|  4|  Meena| 27|31000.0|
|  5|   Ravi| 35|75000.0|
|  6|  Divya| 23|22000.0|
|  7| Suresh| 40|90000.0|
+---+-------+---+-------+



In [8]:
# Filter employees with salary greater than 30,000
print("Employees with Salary > 30,000:")
df.filter(df["Salary"] > 30000).show()

Employees with Salary > 30,000:
+---+-------+---+-------+
| ID|   Name|Age| Salary|
+---+-------+---+-------+
|  1|Karthik| 22|45000.0|
|  3|  Arjun| 30|62000.0|
|  4|  Meena| 27|31000.0|
|  5|   Ravi| 35|75000.0|
|  7| Suresh| 40|90000.0|
+---+-------+---+-------+



In [9]:
# End Spark Session
spark.stop()